# Horticulture Statistics at a Glance-2018 - Question and Answer Dataset

Generated starter notebook from [DesiData](https://www.desidata.in) — India's open dataset catalogue. Run the cells top to bottom.

10 extractive question-and-answer pairs built from Horticulture Statistics at a Glance-2018, published by nhb.gov.in. HORTICULTURAL STATISTICS AT A GLANCE 2018 Government of India Ministry of Agriculture & Farmers’ Welfare Department of Agriculture, Cooperation & Farmers’ Welfare Horticulture Statistics Division Horticulture Statistics Division Department of Agriculture, Cooperation & Farmers' Welfare Ministry of Agriculture & Farmers' Welfare Government of India Horticultural Statistics at a Glance 2018 © Government of India PDES – 256 (E) Controller of Publication 500-2018 – (DSK-III) Price: `695/- Visit us at: www.agricoop.nic.in Horticultural Statistics at a Glance 2018 SANJAY AGARWAL Wt Tt PRRETARY gfe ua Peart aeart sare a, aeaar ce Peart sear fart Government of India wena aa Ministry of Agriculture & Farmers Welfare Department of Agriculture, Cooperation & Farmers Welfare MESSAGE Horticulture sector has become one of the major drivers of growth in the Agriculture sector. Horticulture sector provides employment opportunities across primary, secondary and territory sectors. Horticulture crops, particularly fruit crops are relatively resilient to changes in weather conditions. Vegetables are mostly grown by small and marginal farmers and augments the income of farmers. This sector also enables the population at large to enjoy a diverse and balanced diet for health living .The sector has gained prominence over the last few years contributing a growing share in GVA of the Agriculture and allied sectors. In order to provide impetus to the Horticulture sector, Government has taken several initiatives. The major scheme - Mission for Integrated Development (MIDH) is being implemented in the country by adopting an end to end approach for increasing production of horticultu

Source: nhb.gov.in  
Licence: Not specified  
Dataset page: https://www.desidata.in/datasets/www-nhb-gov-in-183d5ccec6b2b9466bec65ef7b500211

In [ ]:
!pip install pandas matplotlib --quiet

## Load the dataset

Dataset downloads require a free DD token. Create one in Profile & settings, then set `DD_TOKEN` in your environment.
In Colab, add `DD_TOKEN` under Secrets and enable notebook access to that secret.

In [ ]:
import io
import json
import os
import pandas as pd
import urllib.request

url = "https://www.desidata.in/api/datasets/www-nhb-gov-in-183d5ccec6b2b9466bec65ef7b500211/download"
if "COLAB_RELEASE_TAG" in os.environ and "DD_TOKEN" not in os.environ:
    from google.colab import userdata
    os.environ["DD_TOKEN"] = userdata.get("DD_TOKEN")
request = urllib.request.Request(url, headers={"Authorization": f"Bearer {os.environ['DD_TOKEN']}", "Accept": "application/json"})
with urllib.request.urlopen(request) as response:
    file_url = json.load(response)["download_url"]
# Fetch the signed file URL separately; never send DD_TOKEN to the storage host.
with urllib.request.urlopen(file_url) as response:
    df = pd.read_csv(io.BytesIO(response.read()))

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## Explore

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## See one example

Every row pairs a question with the text it was written from.

In [ ]:
row = df.iloc[0]

print("QUESTION:", row["question"])

print("ANSWER:", str(row["answer"])[:400])

print()
print("SOURCE CONTEXT:")
print(str(row["context"])[:600])

## Mini baseline — can TF-IDF find the source paragraph?

Each question was written from its row's source paragraph. A simple TF-IDF
retriever has to rank the correct one first. Its hit rate is the number
any trained model on this dataset must beat.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import numpy as np

sample = df.head(300)
questions = sample["question"].astype(str).to_numpy()
passages = sample["context"].astype(str).to_numpy()

vectorizer = TfidfVectorizer(stop_words="english")
passage_vectors = vectorizer.fit_transform(passages)
question_vectors = vectorizer.transform(questions)

best = np.empty(len(questions), dtype=int)
for i in range(len(questions)):
    best[i] = linear_kernel(question_vectors[i], passage_vectors).argmax()

hit_rate = (passages[best] == passages).mean()
print(f"TF-IDF retrieval lands on the correct paragraph for {hit_rate:.0%} of questions")
print("That is the number a trained model has to beat.")

## What will you build?

- Compare yields or prices across crops, seasons or districts
- Model an outcome column against rainfall or year if they exist
- Combine this with weather or satellite data for a stronger model

- Browse more [Agriculture datasets](https://www.desidata.in/datasets?category=Agriculture) or the [full catalogue](https://www.desidata.in/datasets).
- Sharing your work? Tag DesiData — we feature community notebooks built on this catalogue.

In [ ]:
# Data via DesiData — India's open dataset catalogue
print("Dataset page:", "https://www.desidata.in/datasets/www-nhb-gov-in-183d5ccec6b2b9466bec65ef7b500211")